## 05 — Validação formal, visual e smoke test Ultralytics (v3)
Este notebook valida conjuntos exatos de arquivos, labels OBB em [0,1], integridade por
hash, ausência de leakage e compatibilidade real com o carregador Ultralytics. A aprovação
exige validação automática, revisão visual registrada e smoke test.

In [1]:
%pip install -q opencv-python-headless pandas matplotlib pyyaml ultralytics

from google.colab import drive
from pathlib import Path
import hashlib, json, math, os, shutil, time
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
from tqdm.auto import tqdm

def log_etapa(mensagem):
    print(f"[{time.strftime('%H:%M:%S')}] {mensagem}", flush=True)

def copiar_arvore_com_progresso(origem, destino, descricao):
    origem, destino = Path(origem), Path(destino)
    arquivos = sorted(p for p in origem.rglob('*') if p.is_file())
    log_etapa(f'{descricao}: início ({len(arquivos)} arquivos)')
    inicio = time.perf_counter()
    for arquivo in tqdm(arquivos, desc=descricao, unit='arq'):
        alvo = destino / arquivo.relative_to(origem)
        alvo.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(arquivo, alvo)
    log_etapa(f'{descricao}: concluído em {time.perf_counter()-inicio:.1f}s')

drive.mount('/content/drive', force_remount=True)
BACKUP_DIR=Path('/content/drive/MyDrive/tcc-satellite-streaks')
ponteiro=BACKUP_DIR/'data/synthetic_runs/LATEST_RUN_V3.txt'
RUN_ID=ponteiro.read_text(encoding='utf-8').strip()
DRIVE_RUN=BACKUP_DIR/'data/synthetic_runs'/RUN_ID
RUN=Path('data/synthetic_runs')/RUN_ID
copiar_arvore_com_progresso(DRIVE_RUN, RUN, 'Restaurando dataset sintético')
df=pd.read_csv(RUN/'manifest_geracao_v3.csv')
CONFIG=json.loads((RUN/'config_geracao_v3.json').read_text(encoding='utf-8'))
HASH_MANIFEST=hashlib.sha256((RUN/'manifest_geracao_v3.csv').read_bytes()).hexdigest()
print(RUN_ID,len(df),HASH_MANIFEST)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.7/45.7 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.3/65.3 kB 5.4 MB/s eta 0:00:00
Mounted at /content/drive
[23:39:24] Restaurando dataset sintético: início (12003 arquivos)


Restaurando dataset sintético:   0%|          | 0/12003 [00:00<?, ?arq/s]

[00:26:37] Restaurando dataset sintético: concluído em 2833.8s
synthetic_v3_c41d5091de56 6000 b635aff10e8b24d3729b4b399f50a8223e0d4373d155a7f16bf232c58d844b42


## 1. Arquivos exatos, hashes e imagens válidas

In [2]:
def sha256_arquivo(caminho,bloco=1024*1024):
    h=hashlib.sha256()
    with open(caminho,'rb') as arq:
        for parte in iter(lambda:arq.read(bloco),b''): h.update(parte)
    return h.hexdigest()

problemas=[]
for split in ['train','val','test']:
    esperado=set(df.loc[df.split==split,'id'])
    imgs={p.stem for p in (RUN/'images'/split).glob('*.png')}
    lbls={p.stem for p in (RUN/'labels'/split).glob('*.txt')}
    if imgs!=esperado: problemas.append(f'{split}: imagens faltantes/sobras: -{len(esperado-imgs)} +{len(imgs-esperado)}')
    if lbls!=esperado: problemas.append(f'{split}: labels faltantes/sobras: -{len(esperado-lbls)} +{len(lbls-esperado)}')

for linha in tqdm(df.itertuples(), total=len(df), desc='Validando arquivos e hashes', unit='amostra'):
    imgp=RUN/'images'/linha.split/(linha.id+'.png'); lblp=RUN/'labels'/linha.split/(linha.id+'.txt')
    if imgp.exists() and sha256_arquivo(imgp)!=linha.sha256_imagem: problemas.append(f'{linha.id}: hash da imagem divergente')
    if lblp.exists() and sha256_arquivo(lblp)!=linha.sha256_label: problemas.append(f'{linha.id}: hash do label divergente')
    img=cv2.imread(str(imgp),cv2.IMREAD_GRAYSCALE) if imgp.exists() else None
    if img is None or img.shape!=(CONFIG['tamanho'],CONFIG['tamanho']): problemas.append(f'{linha.id}: imagem inválida')

print('Problemas de arquivos:',len(problemas))
if problemas: print('\n'.join(problemas[:20]))
assert not problemas

Validando arquivos e hashes:   0%|          | 0/6000 [00:00<?, ?amostra/s]

Problemas de arquivos: 0


## 2. Labels YOLO-OBB estritos
Diferentemente da versão anterior, qualquer coordenada fora de [0,1] reprova o dataset.

In [3]:
def area_poligono(p):
    return abs(sum(p[i][0]*p[(i+1)%4][1]-p[(i+1)%4][0]*p[i][1] for i in range(4)))/2

def convexo(p,tol=1e-10):
    cruz=[]
    for i in range(4):
        a,b,c=np.array(p[i]),np.array(p[(i+1)%4]),np.array(p[(i+2)%4])
        cruz.append(np.cross(b-a,c-b))
    return all(v>=-tol for v in cruz) or all(v<=tol for v in cruz)

erros_label=[]; minimo=1.0; maximo=0.0; n_fora=0
for linha in tqdm(df.itertuples(), total=len(df), desc='Validando labels OBB', unit='label'):
    caminho=RUN/'labels'/linha.split/(linha.id+'.txt')
    conteudo=caminho.read_text(encoding='utf-8').strip()
    if linha.tipo=='negativo':
        if conteudo: erros_label.append((linha.id,'negativo com label não vazio'))
        continue
    linhas=[l for l in conteudo.splitlines() if l.strip()]
    if len(linhas)!=1: erros_label.append((linha.id,f'{len(linhas)} objetos, esperado 1')); continue
    tokens=linhas[0].split()
    if len(tokens)!=9: erros_label.append((linha.id,f'{len(tokens)} tokens')); continue
    if tokens[0]!='0': erros_label.append((linha.id,'classe diferente de 0'))
    try: coords=[float(x) for x in tokens[1:]]
    except ValueError: erros_label.append((linha.id,'coordenada não numérica')); continue
    minimo=min(minimo,min(coords)); maximo=max(maximo,max(coords))
    fora=sum(c<0 or c>1 for c in coords); n_fora+=fora
    if fora: erros_label.append((linha.id,'coordenada fora de [0,1]'))
    pontos=[(coords[i],coords[i+1]) for i in range(0,8,2)]
    if area_poligono(pontos)<1e-7: erros_label.append((linha.id,'área degenerada'))
    if not convexo(pontos): erros_label.append((linha.id,'ordem não convexa'))

print(f'Faixa observada: [{minimo:.8f}, {maximo:.8f}] | coordenadas fora: {n_fora}')
print('Erros de label:',len(erros_label))
if erros_label: print(erros_label[:20])
assert not erros_label

Validando labels OBB:   0%|          | 0/6000 [00:00<?, ?label/s]

/tmp/ipykernel_460/1755625127.py:8: DeprecationWarning: Arrays of 2-dimensional vectors are deprecated. Use arrays of 3-dimensional vectors instead. (deprecated in NumPy 2.0)
  cruz.append(np.cross(b-a,c-b))


Faixa observada: [0.00451195, 0.99340310] | coordenadas fora: 0
Erros de label: 0


## 3. Leakage, duplicatas, composição e pseudo-replicação

In [4]:
assert df['id'].is_unique
assert df.groupby('sha256_fundo')['split'].nunique().max()==1
assert not ((df.categoria_fundo=='B')&(df.split!='train')).any()
assert (df.tipo=='positivo').sum()==3000 and (df.tipo=='negativo').sum()==3000

hashes_multi_split=df.groupby('sha256_imagem')['split'].nunique()
duplicatas_cruzadas=hashes_multi_split[hashes_multi_split>1]
assert len(duplicatas_cruzadas)==0,'Imagem sintética idêntica em splits diferentes.'
duplicatas_mesmo_split=df.duplicated(['split','sha256_imagem']).sum()
print('Duplicatas dentro do mesmo split:',duplicatas_mesmo_split)
print('Fundos únicos por split:',df.groupby('split')['sha256_fundo'].nunique().to_dict())
print('\nDistribuição:')
print(pd.crosstab([df.split,df.tipo],df.subtipo))
print('\nCategoria de fundo:')
print(pd.crosstab(df.split,df.categoria_fundo))
print('\nUso mínimo/máximo de um fundo por split:')
print(df.groupby(['split','sha256_fundo']).size().groupby('split').agg(['min','max','median']))

Duplicatas dentro do mesmo split: 0
Fundos únicos por split: {'test': 40, 'train': 245, 'val': 40}

Distribuição:
subtipo         constante  hard_sintetico  limpo  tracejado  variavel
split tipo                                                           
test  negativo          0              90    360          0         0
      positivo        270               0      0         67       113
train negativo          0             420   1680          0         0
      positivo       1260               0      0        315       525
val   negativo          0              90    360          0         0
      positivo        270               0      0         67       113

Categoria de fundo:
categoria_fundo     A    B
split                     
test              900    0
train            3360  840
val               900    0

Uso mínimo/máximo de um fundo por split:
       min  max  median
split                  
test    22   23    22.5
train   11   20    20.0
val     22   23    22.5


## 4. Revisão visual estratificada registrada
A célula produz exatamente 50 itens por split. Depois de inspecionar as três grades,
altere APROVAR_REVISAO_VISUAL para True e execute a célula de registro.

In [5]:
def maior_resto(total,proporcoes):
    bruto={k:total*v for k,v in proporcoes.items()}; base={k:int(np.floor(v)) for k,v in bruto.items()}
    for k in sorted(proporcoes,key=lambda k:(bruto[k]-base[k],k),reverse=True)[:total-sum(base.values())]: base[k]+=1
    return base

def amostra_exata(df_split,n=50):
    tamanhos=df_split.groupby(['tipo','subtipo']).size()
    props={k:v/len(df_split) for k,v in tamanhos.items()}
    cont=maior_resto(n,props); partes=[]
    for chave,quantidade in cont.items():
        g=df_split[(df_split.tipo==chave[0])&(df_split.subtipo==chave[1])]
        partes.append(g.sample(quantidade,random_state=20260823))
    return pd.concat(partes).sample(frac=1,random_state=20260823).reset_index(drop=True)

revisados=[]
for split in ['train','val','test']:
    amostra=amostra_exata(df[df.split==split],50)
    revisados.append(amostra.assign(split_revisado=split))
    fig,axes=plt.subplots(5,10,figsize=(20,11)); axes=axes.flatten()
    for ax,linha in zip(axes,amostra.itertuples()):
        img=cv2.imread(str(RUN/'images'/split/(linha.id+'.png')),cv2.IMREAD_GRAYSCALE)
        ax.imshow(img,cmap='gray',vmin=0,vmax=255)
        texto=(RUN/'labels'/split/(linha.id+'.txt')).read_text().split()
        if len(texto)==9:
            c=[float(x) for x in texto[1:]]; pts=[(c[i]*img.shape[1],c[i+1]*img.shape[0]) for i in range(0,8,2)]
            ax.add_patch(plt.Polygon(pts,fill=False,edgecolor='red',linewidth=.8))
        ax.set_title(f'{linha.tipo[:3]}/{linha.subtipo}\n{linha.fonte_fundo}/{linha.categoria_fundo}',fontsize=5)
        ax.axis('off')
    plt.suptitle(f'{RUN_ID} — {split} — 50 amostras')
    plt.tight_layout(); plt.savefig(RUN/f'overlay_validacao_{split}.png',dpi=160,bbox_inches='tight'); plt.show(); plt.close(fig)
df_revisados=pd.concat(revisados,ignore_index=True)
df_revisados[['id','split','tipo','subtipo','fundo_origem','sha256_imagem']].to_csv(RUN/'amostra_revisao_visual.csv',index=False)

Output hidden; open in https://colab.research.google.com to view.

In [6]:
from google.colab import files
from zipfile import ZipFile, ZIP_DEFLATED
from pathlib import Path

nomes = [
    'overlay_validacao_train.png',
    'overlay_validacao_val.png',
    'overlay_validacao_test.png',
    'amostra_revisao_visual.csv',
]

pacote = Path('/content/revisao_visual_v3.zip')

with ZipFile(pacote, 'w', ZIP_DEFLATED) as zipf:
    for nome in nomes:
        caminho = RUN / nome
        assert caminho.is_file(), f'Arquivo ausente: {caminho}'
        zipf.write(caminho, arcname=nome)

files.download(str(pacote))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [7]:
for split in ['train', 'val', 'test']:
    amostra = (
        df_revisados[df_revisados.split_revisado == split]
        .reset_index(drop=True)
    )

    fig, axes = plt.subplots(5, 10, figsize=(20, 11))
    axes = axes.flatten()

    for ax, linha in zip(axes, amostra.itertuples()):
        img = cv2.imread(
            str(RUN / 'images' / split / f'{linha.id}.png'),
            cv2.IMREAD_GRAYSCALE
        )

        ax.imshow(img, cmap='gray', vmin=0, vmax=255)
        ax.set_title(
            f'{linha.tipo[:3]}/{linha.subtipo}\n'
            f'{linha.fonte_fundo}/{linha.categoria_fundo}',
            fontsize=5
        )
        ax.axis('off')

    plt.suptitle(f'{RUN_ID} — {split} — 50 amostras — sem overlay')
    plt.tight_layout()

    destino = RUN / f'raw_validacao_{split}.png'
    plt.savefig(destino, dpi=160, bbox_inches='tight')
    plt.show()
    plt.close(fig)

print('Grades sem overlay geradas.')

Output hidden; open in https://colab.research.google.com to view.

In [8]:
from google.colab import files
from zipfile import ZipFile, ZIP_DEFLATED
from pathlib import Path

nomes = [
    'raw_validacao_train.png',
    'raw_validacao_val.png',
    'raw_validacao_test.png',
    'overlay_validacao_train.png',
    'overlay_validacao_val.png',
    'overlay_validacao_test.png',
    'amostra_revisao_visual.csv',
]

pacote = Path('/content/revisao_visual_v3_completa.zip')

with ZipFile(pacote, 'w', ZIP_DEFLATED) as zipf:
    for nome in nomes:
        caminho = RUN / nome
        assert caminho.is_file(), f'Arquivo ausente: {caminho}'
        zipf.write(caminho, arcname=nome)

files.download(str(pacote))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
APROVAR_REVISAO_VISUAL=True  # mude para True apenas depois de examinar as três grades
if APROVAR_REVISAO_VISUAL:
    registro={'run_id':RUN_ID,'sha256_manifest':HASH_MANIFEST,'aprovado':True,
              'sha256_amostra':sha256_arquivo(RUN/'amostra_revisao_visual.csv')}
    (RUN/'revisao_visual_aprovada.json').write_text(json.dumps(registro,indent=2),encoding='utf-8')
    artefatos_visual = [
        RUN/'revisao_visual_aprovada.json', RUN/'amostra_revisao_visual.csv',
        RUN/'overlay_validacao_train.png', RUN/'overlay_validacao_val.png',
        RUN/'overlay_validacao_test.png'
    ]
    for arquivo in tqdm(artefatos_visual, desc='Salvando revisão visual', unit='arq'):
        shutil.copy2(arquivo, DRIVE_RUN/arquivo.name)
    print('Revisão visual registrada.')
else:
    print('Revisão visual ainda não aprovada.')

Salvando revisão visual:   0%|          | 0/5 [00:00<?, ?arq/s]

Revisão visual registrada.


## 5. Smoke test real do Ultralytics
O teste usa apenas uma fração do treino e validação. Ele existe para confirmar que o
carregador aceita todos os formatos; não produz resultado científico.

In [12]:
dataset_yaml={
    'path':str(RUN.resolve()),'train':'images/train','val':'images/val','test':'images/test',
    'names':{0:'satellite_streak'}
}
yaml_path=RUN/'dataset_v3.yaml'
yaml_path.write_text(yaml.safe_dump(dataset_yaml,sort_keys=False),encoding='utf-8')

PROTOCOLO_SMOKE='balanced_train_val_v2'
EXECUTAR_SMOKE_ULTRALYTICS=True  # mude para True quando a validação anterior estiver limpa
if EXECUTAR_SMOKE_ULTRALYTICS:
    import torch
    from ultralytics import YOLO

    # Não use ``fraction`` sobre os nomes neg_*/pos_*: o fatiamento ordenado pode
    # selecionar somente negativos e produzir um falso positivo no gate.
    grupos=[('positivo','constante'),('positivo','variavel'),('positivo','tracejado'),
            ('negativo','limpo'),('negativo','hard_sintetico')]

    def selecionar_smoke(split,n_por_grupo=2):
        partes=[]
        for indice,(tipo,subtipo) in enumerate(grupos):
            candidatos=df[(df.split==split)&(df.tipo==tipo)&(df.subtipo==subtipo)]
            assert len(candidatos)>=n_por_grupo,f'Sem amostras suficientes para {split}/{tipo}/{subtipo}'
            partes.append(candidatos.sample(n_por_grupo,random_state=20260824+indice))
        amostra=pd.concat(partes,ignore_index=True).sort_values('id').reset_index(drop=True)
        for linha in amostra.itertuples():
            tokens=(RUN/'labels'/split/(linha.id+'.txt')).read_text(encoding='utf-8').split()
            esperado=9 if linha.tipo=='positivo' else 0
            assert len(tokens)==esperado,f'Label inesperado no smoke: {linha.id}'
        return amostra

    smoke_dir=RUN/'smoke_protocol_v2'
    smoke_dir.mkdir(parents=True,exist_ok=True)
    smoke_amostras={split:selecionar_smoke(split) for split in ['train','val']}
    for split,amostra in smoke_amostras.items():
        caminhos=[str((RUN/'images'/split/(id_amostra+'.png')).resolve()) for id_amostra in amostra.id]
        (smoke_dir/f'{split}.txt').write_text('\n'.join(caminhos)+'\n',encoding='utf-8')

    smoke_yaml={
        'path':str(RUN.resolve()),
        'train':str((smoke_dir/'train.txt').resolve()),
        'val':str((smoke_dir/'val.txt').resolve()),
        'names':{0:'satellite_streak'}
    }
    smoke_yaml_path=smoke_dir/'dataset_smoke_balanced_v2.yaml'
    smoke_yaml_path.write_text(yaml.safe_dump(smoke_yaml,sort_keys=False),encoding='utf-8')
    print('Amostra balanceada do smoke:')
    print(pd.concat(smoke_amostras.values(),ignore_index=True).groupby(['split','tipo','subtipo']).size())

    projeto_smoke=(RUN/'smoke_ultralytics').resolve()
    modelo=YOLO('yolo11n-obb.pt')
    modelo.train(data=str(smoke_yaml_path.resolve()),epochs=1,imgsz=256,batch=4,
                 workers=2,device=0 if torch.cuda.is_available() else 'cpu',
                 project=str(projeto_smoke),name='loader_check_balanced_v2',exist_ok=True,
                 plots=False,verbose=True)
    save_dir=Path(modelo.trainer.save_dir).resolve()
    assert save_dir==projeto_smoke/'loader_check_balanced_v2','Ultralytics salvou fora do diretório esperado.'
    best=save_dir/'weights'/'best.pt'; last=save_dir/'weights'/'last.pt'
    assert best.is_file() and last.is_file(),'Pesos do smoke test não foram produzidos.'

    registro={
        'run_id':RUN_ID,'sha256_manifest':HASH_MANIFEST,'aprovado':True,
        'modelo':'yolo11n-obb.pt','protocolo':PROTOCOLO_SMOKE,
        'treino':{'imagens':len(smoke_amostras['train']),
                  'positivas':int((smoke_amostras['train'].tipo=='positivo').sum()),
                  'negativas':int((smoke_amostras['train'].tipo=='negativo').sum())},
        'validacao':{'imagens':len(smoke_amostras['val']),
                     'positivas':int((smoke_amostras['val'].tipo=='positivo').sum()),
                     'negativas':int((smoke_amostras['val'].tipo=='negativo').sum())},
        'sha256_best_pt':sha256_arquivo(best)
    }
    tmp=RUN/'smoke_ultralytics_ok.json.tmp'
    tmp.write_text(json.dumps(registro,indent=2),encoding='utf-8')
    tmp.replace(RUN/'smoke_ultralytics_ok.json')
    shutil.copy2(yaml_path, DRIVE_RUN/yaml_path.name)
    copiar_arvore_com_progresso(smoke_dir, DRIVE_RUN/'smoke_protocol_v2',
                                'Salvando contrato do smoke test')
    copiar_arvore_com_progresso(projeto_smoke, DRIVE_RUN/'smoke_ultralytics',
                                'Salvando resultados do smoke test')
    shutil.copy2(RUN/'smoke_ultralytics_ok.json', DRIVE_RUN/'smoke_ultralytics_ok.json')
    print(json.dumps(registro,indent=2,ensure_ascii=False))
    print('Smoke test balanceado aprovado.')
else:
    print('Smoke test não executado nesta célula.')

Amostra balanceada do smoke:
split  tipo      subtipo       
train  negativo  hard_sintetico    2
                 limpo             2
       positivo  constante         2
                 tracejado         2
                 variavel          2
val    negativo  hard_sintetico    2
                 limpo             2
       positivo  constante         2
                 tracejado         2
                 variavel          2
dtype: int64
Ultralytics 8.4.127 🚀 Python-3.13.15 torch-2.11.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data/synthetic_runs/synthetic_v3_c41d5091de56/smoke_protocol_v2/dataset_smoke_balanced_v2.yaml, degrees=0.0, deterministic=True, d

Salvando contrato do smoke test:   0%|          | 0/3 [00:00<?, ?arq/s]

[00:56:19] Salvando contrato do smoke test: concluído em 0.1s
[00:56:19] Salvando resultados do smoke test: início (4 arquivos)


Salvando resultados do smoke test:   0%|          | 0/4 [00:00<?, ?arq/s]

[00:56:20] Salvando resultados do smoke test: concluído em 0.2s
{
  "run_id": "synthetic_v3_c41d5091de56",
  "sha256_manifest": "b635aff10e8b24d3729b4b399f50a8223e0d4373d155a7f16bf232c58d844b42",
  "aprovado": true,
  "modelo": "yolo11n-obb.pt",
  "protocolo": "balanced_train_val_v2",
  "treino": {
    "imagens": 10,
    "positivas": 6,
    "negativas": 4
  },
  "validacao": {
    "imagens": 10,
    "positivas": 6,
    "negativas": 4
  },
  "sha256_best_pt": "6aaa6938264b354d6b70aade8d9702d6c6b00028129db8a7c6c57206fab84e1d"
}
Smoke test balanceado aprovado.


## 6. Gate final da versão v3

In [13]:
PROTOCOLO_SMOKE_ESPERADO='balanced_train_val_v2'

def aprovacao_valida(nome,protocolo=None):
    caminho=RUN/nome
    if not caminho.exists(): return False
    registro=json.loads(caminho.read_text(encoding='utf-8'))
    base=registro.get('aprovado') is True and registro.get('sha256_manifest')==HASH_MANIFEST
    return base and (protocolo is None or registro.get('protocolo')==protocolo)

visual_ok=aprovacao_valida('revisao_visual_aprovada.json')
smoke_ok=aprovacao_valida('smoke_ultralytics_ok.json',PROTOCOLO_SMOKE_ESPERADO)
resumo={'run_id':RUN_ID,'sha256_manifest':HASH_MANIFEST,'arquivos_ok':True,'labels_ok':True,
        'leakage_ok':True,'revisao_visual_ok':visual_ok,'smoke_ultralytics_ok':smoke_ok,
        'aprovado_para_treino':bool(visual_ok and smoke_ok)}
(RUN/'resumo_validacao_v3.json').write_text(json.dumps(resumo,indent=2),encoding='utf-8')
shutil.copy2(RUN/'resumo_validacao_v3.json', DRIVE_RUN/'resumo_validacao_v3.json')
print(json.dumps(resumo,indent=2,ensure_ascii=False))
assert resumo['aprovado_para_treino'],'Dataset ainda não passou por revisão visual e smoke test registrados.'

{
  "run_id": "synthetic_v3_c41d5091de56",
  "sha256_manifest": "b635aff10e8b24d3729b4b399f50a8223e0d4373d155a7f16bf232c58d844b42",
  "arquivos_ok": true,
  "labels_ok": true,
  "leakage_ok": true,
  "revisao_visual_ok": true,
  "smoke_ultralytics_ok": true,
  "aprovado_para_treino": true
}
